# Composition, Refactoring, and Method Checks


```{contents}
:local:
:depth: 2
```


```{index} Refactor in Small Steps That Keep an Observable Contract
```



```{index} Refactor in Small Steps That Keep an Observable Contract
```

## Refactor in Small Steps That Keep an Observable Contract



An invoice caller repeats price-times-quantity arithmetic in several places. Extracting LineTotal can give that calculation one name, but a safe refactoring needs more than moving those characters into braces. First record the existing accepted inputs, numerical outputs, formatting and mutation behavior. Then choose the new method's inputs/result so that callers can express the same work without depending on its internal temporary variables.

For price 12 and quantity three, independently calculate 36 before running either version. A fixed packing charge of four makes the order total 40. Keep these two contracts distinct: LineTotal returns 36; OrderTotal returns 40. If the extracted line calculation starts adding packing as well, a caller that already adds packing can charge it twice even though both helper methods contain plausible arithmetic.

| Refactoring stage | Evidence to retain |
| --- | --- |
| Before extraction | Expected amount and current observable output |
| After extracting LineTotal | Same line amount for ordinary and zero inputs |
| After composing OrderTotal | Exactly one application of the fixed charge |
| After formatting a report | Same numerical meaning; display policy stays explicit |

A useful composition passes returned values between helpers. It does not make each helper prompt independently for the same data. If CalculateTotal reads a second quantity while the caller formats the first, the report can display one order and charge another. Keep interaction in the caller and pass the accepted values into calculation helpers.

For related explanation, see the further reading below.[^development-reading]


```{index} Choose Tests That Can Distinguish Plausible Defects
```



```{index} Choose Tests That Can Distinguish Plausible Defects
```

## Choose Tests That Can Distinguish Plausible Defects



A zero-quantity order exposes a fixed-charge rule: the line amount is zero but the order amount may still be four. That result follows from this example's contract, not a universal claim that every business must charge empty orders. If a later policy waives packing for empty orders, revise the contract and the independent expectations before revising the method.

A test with price zero and nonzero quantity checks a different route to a zero line total. A normal nonzero case checks multiplication. A changed quantity checks that a hard-coded result is not being returned. Expected values should be derived from the requirement rather than copied from the candidate implementation's output; otherwise a defective method can teach its own test what to accept.

Guided synthesis: write a report helper that calls LineTotal and returns text without printing. Keep the item description and numerical result visible in its returned sentence. Then change only the display format. Explain why calculation tests can remain unchanged even when the report's string test changes. This separation prepares the repeatable method tests and debugging workflows of Chapter 6.


```{index} Refactor and Check a Checkout Calculation
```



```{index} Refactor and Check a Checkout Calculation
```

(method-synthesis)=

## Refactor and Check a Checkout Calculation




```{index} Refactor and Check a Checkout Calculation
```


Assume a nonnegative unit price and quantity. The original program calculates one line and adds a fixed packing charge; its two orders duplicate that policy. Refactoring should preserve the numerical behavior while giving separate names to line calculation and packing. All zero-quantity orders in this example still pay the packing charge; a different policy would require a changed contract, not just a code cleanup.


In [1]:
using System;
decimal first = 12m * 3 + 4m;
decimal second = 5m * 2 + 4m;
Console.WriteLine($"Before: {first:F2}, {second:F2}");
Console.WriteLine($"After: {OrderTotal(12m, 3):F2}, {OrderTotal(5m, 2):F2}");
static decimal LineTotal(decimal unitPrice, int quantity)
{
    return unitPrice * quantity;
}
static decimal OrderTotal(decimal unitPrice, int quantity)
{
    return LineTotal(unitPrice, quantity) + 4m;
}


Before: 40.00, 14.00
After: 40.00, 14.00


Both versions print 40.00 and 14.00. OrderTotal calls LineTotal, receives its returned value, then adds packing. The console formatting stays in the caller. The first method does not prompt for a quantity or silently modify another method's locals.

Derive expected results from the requirement, not by copying the code's output. A test of 12 × 3 + 4 expects 40; a zero quantity expects 4 under this stated policy. A second ordinary case helps reveal a hard-coded answer.

| Unit price | Quantity | Expected total | Reason |
| --- | --- | --- | --- |
| 12.00 | 3 | 40.00 | 36.00 plus 4.00 packing |
| 5.00 | 2 | 14.00 | 10.00 plus packing |
| 12.00 | 0 | 4.00 | Zero line value, packing remains |
| 0.00 | 1 | 4.00 | Zero-price item under this contract |

A useful check reports expected, actual, and success, rather than merely printing a number and hoping it looks plausible. This is a testing habit; a testing framework is not required yet.


In [1]:
using System;
Check(12m, 3, 40m);
Check(5m, 2, 14m);
Check(12m, 0, 4m);
Check(0m, 1, 4m);
static decimal LineTotal(decimal unitPrice, int quantity) { return unitPrice * quantity; }
static decimal OrderTotal(decimal unitPrice, int quantity) { return LineTotal(unitPrice, quantity) + 4m; }
static void Check(decimal unitPrice, int quantity, decimal expected)
{
    decimal actual = OrderTotal(unitPrice, quantity);
    Console.WriteLine($"Expected {expected:F2}; actual {actual:F2}; pass={actual == expected}");
}


Expected 40.00; actual 40.00; pass=True
Expected 14.00; actual 14.00; pass=True
Expected 4.00; actual 4.00; pass=True
Expected 4.00; actual 4.00; pass=True


All four checks report True. Deliberately omit the packing charge and predict which checks fail before running the variation. Testing a few values supports a contract; it does not prove correctness for every possible input. Negative inputs remain outside this exercise's domain and need an explicit future policy.

**Common mistakes:** printing inside a helper intended only to calculate; returning a formatted string when the caller needs a number; using a caller variable instead of a parameter; and testing only one convenient input. The lab and original restored exercises apply these distinctions after they have been taught here.


```{index} Guided Synthesis: Report a Line Total
```



```{index} Guided Synthesis: Report a Line Total
```

## Guided Synthesis: Report a Line Total




```{index} Guided Synthesis: Report a Line Total
```


Define LineTotal and LineDescription. The description helper must call LineTotal, then return a formatted string without printing. Print the description in the caller and check two independently computed numerical totals. Extend to zero quantity under the stated nonnegative-input contract.


In [ ]:
// Exercise: Guided Synthesis: Report a Line Total
// exercise-id: cscs-methods-guided-m5-751ba022
using System;
Console.WriteLine(LineDescription("PEN", 2m, 3));
Console.WriteLine($"Zero quantity: {LineTotal(2m, 0):F2}");
static decimal LineTotal(decimal price, int quantity)
{
    // Your code starts here.
    return 0m;
    // Your code ends here.
}
static string LineDescription(string code, decimal price, int quantity)
{
    // Your code starts here.
    return "";
    // Your code ends here.
}


In [1]:
// Solution: Guided Synthesis: Report a Line Total
using System;

Console.WriteLine(LineDescription("PEN", 2m, 3));
Console.WriteLine($"Zero quantity: {LineTotal(2m, 0):F2}");
static decimal LineTotal(decimal price, int quantity) { return price * quantity; }
static string LineDescription(string code, decimal price, int quantity)
{
    decimal total = LineTotal(price, quantity);
    return $"{code}: {quantity} units, total {total:F2}";
}


PEN: 3 units, total 6.00
Zero quantity: 0.00


```{index} Bounded Extensions: Overloading and Recursion
```



```{index} Bounded Extensions: Overloading and Recursion
```

## Bounded Extensions: Overloading and Recursion




```{index} Bounded Extensions: Overloading and Recursion
```


These are recognition-level extensions, not prerequisites for the required method tasks. In a class, overloading permits the same member name with distinct parameter signatures; changing only the return type does not create a distinct overload. Local functions in the same local scope cannot be overloaded this way. Recursion means a method calls itself; it needs a base case and a progress argument. The supporting reading {doc}`Recursive Methods <0506_recursive_methods>` introduces it with a countdown, a sum, and GCD, and Chapters 17 and 22 develop it for trees and divide-and-conquer sorting. The preview's extension items and the original homework research questions should not be used to assess untaught core skills.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [Loyola, Function Parameters](https://introcs.cs.luc.edu/functions/funcparam.html). The worked business scenarios and guided reasoning here are original teaching examples.
